# 02 — DG38 Detection Baselines & 3-Seed Experiments (YOLO11s / YOLO11n / RT-DETR / Deformable DETR)

**Quy tắc khoa học bắt buộc (Giai đoạn 4 & 5):**
1. **Dữ liệu cố định:**
   - Train: `WasteDetect_Dataset/train` (**8,289 ảnh / 18,345 boxes**, 5 lớp ID 0–4).
   - Validation sạch: `WasteDetect_Dataset/valid_clean_candidate` (**1,743 ảnh / 3,932 boxes**). **Tuyệt đối không dùng tập `valid` gốc (`2,316` ảnh, có `506` ảnh trùng SHA-256 và `28` ảnh trùng pHash với `train`).**
2. **Đánh giá công bằng bằng `COCOeval`:**
   - Mọi detector (`yolo11s`, `yolo11n`, `rtdetr-l`, `deformable_detr`) đều được xuất dự đoán chuẩn COCO JSON và chấm bằng `pycocotools.cocoeval.COCOeval`.
3. **Ngưỡng vận hành độc lập (Model-specific Operating Threshold):**
   - Tìm ngưỡng confidence tối ưu F1 trực tiếp trên **Clean Validation (`1,743` ảnh)** cho từng mô hình, lưu vào `locked_validation_threshold.json`.
4. **Kiểm soát ngân sách CU:**
   - Luôn chạy Pilot đo thời gian & kiểm tra cổng ngân sách ở Cell 3 trước khi bật `RUN_FULL_TRAINING = True` ở Cell 4.

In [ ]:
# [CELL 1] Thiết lập môi trường & tham số thí nghiệm Detection
import os
import sys
import zipfile
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

DRIVE_ROOT = Path(os.environ.get("DG38_DRIVE_ROOT", "/content/drive/MyDrive/DG38_Colab_Package"))
REPO_ROOT = DRIVE_ROOT / "code" if (DRIVE_ROOT / "code" / "model.py").exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXP_DIR = REPO_ROOT / "evaluation_runs" / "dg38_additional_experiments"
DET_ROOT = Path(os.environ.get("DG38_DET_ROOT", "/content/dg38_data/detection/WasteDetect_Dataset"))
YOLO_DATA_DIR = Path("/content/dg38_data/yolo_clean_val")
ARTIFACTS_DIR = (DRIVE_ROOT / "artifacts") if DRIVE_ROOT.exists() else EXP_DIR

# ================= THAM SỐ THÍ NGHIỆM DETECTION =================
ARCH = "yolo11s"                 # Lựa chọn: "yolo11s" | "yolo11n" | "rtdetr-l" | "deformable_detr"
SEED = 42                        # Lựa chọn: 42 | 123 | 2026
EPOCHS = 30                      # Số epochs huấn luyện
IMGSZ = 640 if ARCH != "deformable_detr" else 518

# ================= CỔNG KIỂM SOÁT NGÂN SÁCH CU =================
OBSERVED_CU_PER_HOUR = None      # <-- BẮT BUỘC: Nhập Usage Rate thực tế từ giao diện Colab (ví dụ: 1.48)
BUDGET_CATEGORY = "yolo11s_baseline" if (ARCH.startswith("yolo") and SEED == 42) else "three_seed_experiments"
RUN_FULL_TRAINING = False        # ĐỔI THÀNH True SAU KHI XÁC NHẬN DỰ BÁO CU Ở CELL 3 ĐẠT APPROVED
RUN_LOCKED_TEST_EVAL = False     # ĐỔI THÀNH True Ở CELL 5 SAU KHI ĐÃ TRAIN VÀ KHOÁ THRESHOLD TRÊN CLEAN VAL

if OBSERVED_CU_PER_HOUR is None or float(OBSERVED_CU_PER_HOUR) <= 0:
    raise ValueError("Vui lòng nhập OBSERVED_CU_PER_HOUR (Usage Rate từ 'Runtime -> View resources') trước khi chạy!")

print(f"DET_ROOT      : {DET_ROOT} (exists={DET_ROOT.exists()})")
print(f"YOLO_DATA_DIR : {YOLO_DATA_DIR}")

In [ ]:
# [CELL 2] Chuẩn bị dữ liệu YOLO từ COCO Annotations & Kiểm tra tính hợp lệ của Bounding Boxes (8,289 Train + 1,743 Clean Val)
det_script = EXP_DIR / "dg38_det_trainer.py"
out_dir = ARTIFACTS_DIR / "runs" / f"det_{ARCH}_seed{SEED}"

if DET_ROOT.exists() and ARCH != "deformable_detr":
    cmd_prep = [
        sys.executable,
        str(det_script),
        "--mode", "prepare-yolo",
        "--arch", ARCH,
        "--det-root", str(DET_ROOT),
        "--yolo-work-dir", str(YOLO_DATA_DIR),
        "--output-dir", str(out_dir),
    ]
    res = subprocess.run(cmd_prep, text=True)
    if res.returncode != 0:
        raise RuntimeError("Chuẩn bị dữ liệu YOLO gặp lỗi.")
else:
    print(f"[INFO] DET_ROOT={DET_ROOT} (exists={DET_ROOT.exists()}), ARCH={ARCH}")

In [ ]:
# [CELL 3] Chạy Pilot Đo Tốc Độ Thực Tế & Dự Báo Chi Phí CU
if DET_ROOT.exists():
    cmd_pilot = [
        sys.executable,
        str(det_script),
        "--mode", "pilot",
        "--arch", ARCH,
        "--seed", str(SEED),
        "--epochs", str(EPOCHS),
        "--imgsz", str(IMGSZ),
        "--det-root", str(DET_ROOT),
        "--yolo-work-dir", str(YOLO_DATA_DIR),
        "--output-dir", str(out_dir),
        "--cu-per-hour", str(OBSERVED_CU_PER_HOUR),
        "--budget-category", BUDGET_CATEGORY,
    ]
    res = subprocess.run(cmd_pilot, text=True)
    if res.returncode != 0:
        raise RuntimeError("Pilot Detection gặp lỗi.")

In [ ]:
# [CELL 4] Huấn luyện đầy đủ, Đánh giá bằng COCOeval trên Clean Validation (1,743 ảnh) & Khoá Threshold riêng
if not RUN_FULL_TRAINING:
    print("[DỪNG AN TOÀN] RUN_FULL_TRAINING đang là False. Hãy kiểm tra bảng dự báo CU ở Cell 3 trước khi bật True.")
else:
    cmd_train = [
        sys.executable,
        str(det_script),
        "--mode", "train",
        "--arch", ARCH,
        "--seed", str(SEED),
        "--epochs", str(EPOCHS),
        "--imgsz", str(IMGSZ),
        "--det-root", str(DET_ROOT),
        "--yolo-work-dir", str(YOLO_DATA_DIR),
        "--output-dir", str(out_dir),
        "--cu-per-hour", str(OBSERVED_CU_PER_HOUR),
        "--budget-category", BUDGET_CATEGORY,
        "--resume",
    ]
    res = subprocess.run(cmd_train, text=True)
    if res.returncode != 0:
        raise RuntimeError("Huấn luyện Detection gặp lỗi hoặc bị chặn bởi cổng ngân sách CU.")

In [ ]:
# [CELL 5] Đánh giá Mô hình & Ngưỡng Đã Khoá trên Tập Test Đã Kiểm Toán (871 ảnh / 1,884 boxes)
if not RUN_LOCKED_TEST_EVAL:
    print("[DỪNG AN TOÀN] RUN_LOCKED_TEST_EVAL đang là False. Chỉ bật True sau khi Cell 4 đã hoàn tất và tạo locked_validation_threshold.json.")
else:
    test_zip = DRIVE_ROOT / "zips" / "dg38_det_test_locked.zip"
    test_coco = DET_ROOT / "test_final_candidate" / "_annotations.coco.json"
    if test_zip.exists() and not test_coco.exists():
        print(f"Đang giải nén gói Test Detection {test_zip} -> {DET_ROOT.parent} ...")
        with zipfile.ZipFile(test_zip, "r") as zf:
            zf.extractall(DET_ROOT.parent)

    cmd_eval_test = [
        sys.executable,
        str(det_script),
        "--mode", "eval-test",
        "--arch", ARCH,
        "--seed", str(SEED),
        "--imgsz", str(IMGSZ),
        "--det-root", str(DET_ROOT),
        "--yolo-work-dir", str(YOLO_DATA_DIR),
        "--output-dir", str(out_dir),
    ]
    res = subprocess.run(cmd_eval_test, text=True)
    if res.returncode != 0:
        raise RuntimeError("Đánh giá trên tập Test Detection gặp lỗi.")